# Análisis Visual de MoodMaps

## Patrones emocionales en tiempo real
Análisis completo y visual de los mapas de estados de ánimo de los usuarios

In [ ]:
# Librerías para análisis visual de MoodMaps
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np
import pickle
import joblib
import os

# Configuración para gráficas
plt.style.use('default')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("Listo para analizar MoodMaps")

In [ ]:
# Cargar datos de MoodMaps
df_mood = pd.read_csv('moodmaps.csv')

print(f"Datos cargados: {len(df_mood)} registros de {df_mood['usuario_id'].nunique()} usuarios")
print(f"Período: {df_mood['dia'].min()} - {df_mood['dia'].max()} días")
print(f"Variables emocionales: felicidad, estres, motivacion")

df_mood.head()

In [ ]:
# GRÁFICA 1: Tendencias emocionales por usuario
plt.figure(figsize=(15, 10))

# Subplot 1: Felicidad por día
plt.subplot(3, 2, 1)
for usuario in df_mood['nombre'].unique():
    data = df_mood[df_mood['nombre'] == usuario].groupby('dia')['felicidad'].mean()
    plt.plot(data.index, data.values, marker='o', label=usuario, linewidth=3)
plt.title('Evolución de Felicidad', fontsize=14, fontweight='bold')
plt.xlabel('Día')
plt.ylabel('Nivel de Felicidad')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: Estrés por día
plt.subplot(3, 2, 2)
for usuario in df_mood['nombre'].unique():
    data = df_mood[df_mood['nombre'] == usuario].groupby('dia')['estres'].mean()
    plt.plot(data.index, data.values, marker='s', label=usuario, linewidth=3)
plt.title('Evolución de Estrés', fontsize=14, fontweight='bold')
plt.xlabel('Día')
plt.ylabel('Nivel de Estrés')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 3: Motivación por día
plt.subplot(3, 2, 3)
for usuario in df_mood['nombre'].unique():
    data = df_mood[df_mood['nombre'] == usuario].groupby('dia')['motivacion'].mean()
    plt.plot(data.index, data.values, marker='^', label=usuario, linewidth=3)
plt.title('Evolución de Motivación', fontsize=14, fontweight='bold')
plt.xlabel('Día')
plt.ylabel('Nivel de Motivación')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 4: Distribución por horas
plt.subplot(3, 2, 4)
df_mood['hora'] = pd.to_datetime(df_mood['timestamp']).dt.hour
registro_horas = df_mood['hora'].value_counts().sort_index()
plt.bar(registro_horas.index, registro_horas.values, color='lightblue', alpha=0.7)
plt.title('Registros por Hora del Día', fontsize=14, fontweight='bold')
plt.xlabel('Hora')
plt.ylabel('Número de Registros')

# Subplot 5: Promedio emocional por usuario
plt.subplot(3, 2, 5)
promedios = df_mood.groupby('nombre')[['felicidad', 'estres', 'motivacion']].mean()
promedios.plot(kind='bar', ax=plt.gca(), width=0.8)
plt.title('Promedio Emocional por Usuario', fontsize=14, fontweight='bold')
plt.xticks(rotation=0)
plt.legend()

# Subplot 6: Correlaciones emocionales
plt.subplot(3, 2, 6)
correlaciones = df_mood[['felicidad', 'estres', 'motivacion']].corr()
sns.heatmap(correlaciones, annot=True, cmap='RdYlBu_r', center=0, 
            square=True, fmt='.2f', cbar_kws={'label': 'Correlación'})
plt.title('Correlaciones Emocionales', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("Principales hallazgos:")
felicidad_promedio = df_mood['felicidad'].mean()
estres_promedio = df_mood['estres'].mean()
motivacion_promedio = df_mood['motivacion'].mean()
print(f"Felicidad promedio global: {felicidad_promedio:.2f}")
print(f"Estrés promedio global: {estres_promedio:.2f}")
print(f"Motivación promedio global: {motivacion_promedio:.2f}")
print(f"Usuario más registros: {df_mood.groupby('nombre').size().idxmax()}")
print(f"Hora más activa: {registro_horas.idxmax()}:00h")

In [ ]:
# GRÁFICA 2: Análisis de patrones emocionales
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Scatter plot: Felicidad vs Estrés
scatter_colors = ['red', 'blue', 'green']
usuarios = df_mood['nombre'].unique()
for i, usuario in enumerate(usuarios):
    data = df_mood[df_mood['nombre'] == usuario]
    axes[0,0].scatter(data['felicidad'], data['estres'], 
                     c=scatter_colors[i], label=usuario, alpha=0.7, s=60)
axes[0,0].set_xlabel('Felicidad')
axes[0,0].set_ylabel('Estrés')
axes[0,0].set_title('Relación Felicidad-Estrés por Usuario', fontweight='bold')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 2. Scatter plot: Motivación vs Estrés
for i, usuario in enumerate(usuarios):
    data = df_mood[df_mood['nombre'] == usuario]
    axes[0,1].scatter(data['motivacion'], data['estres'], 
                     c=scatter_colors[i], label=usuario, alpha=0.7, s=60)
axes[0,1].set_xlabel('Motivación')
axes[0,1].set_ylabel('Estrés')
axes[0,1].set_title('Relación Motivación-Estrés por Usuario', fontweight='bold')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# 3. Distribución de felicidad
axes[0,2].hist(df_mood['felicidad'], bins=15, alpha=0.7, color='gold', edgecolor='black')
axes[0,2].axvline(df_mood['felicidad'].mean(), color='red', linestyle='--', linewidth=2, label='Media')
axes[0,2].set_xlabel('Nivel de Felicidad')
axes[0,2].set_ylabel('Frecuencia')
axes[0,2].set_title('Distribución de Felicidad', fontweight='bold')
axes[0,2].legend()
axes[0,2].grid(True, alpha=0.3)

# 4. Distribución de estrés
axes[1,0].hist(df_mood['estres'], bins=15, alpha=0.7, color='orangered', edgecolor='black')
axes[1,0].axvline(df_mood['estres'].mean(), color='blue', linestyle='--', linewidth=2, label='Media')
axes[1,0].set_xlabel('Nivel de Estrés')
axes[1,0].set_ylabel('Frecuencia')
axes[1,0].set_title('Distribución de Estrés', fontweight='bold')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# 5. Distribución de motivación
axes[1,1].hist(df_mood['motivacion'], bins=15, alpha=0.7, color='limegreen', edgecolor='black')
axes[1,1].axvline(df_mood['motivacion'].mean(), color='purple', linestyle='--', linewidth=2, label='Media')
axes[1,1].set_xlabel('Nivel de Motivación')
axes[1,1].set_ylabel('Frecuencia')
axes[1,1].set_title('Distribución de Motivación', fontweight='bold')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

# 6. Estado emocional compuesto
df_mood['bienestar'] = (df_mood['felicidad'] + df_mood['motivacion']) / 2 - df_mood['estres'] / 2
bienestar_diario = df_mood.groupby('dia')['bienestar'].mean()
axes[1,2].plot(bienestar_diario.index, bienestar_diario.values, 
               marker='o', linewidth=3, markersize=8, color='purple')
axes[1,2].fill_between(bienestar_diario.index, bienestar_diario.values, alpha=0.3, color='purple')
axes[1,2].set_xlabel('Día')
axes[1,2].set_ylabel('Índice de Bienestar')
axes[1,2].set_title('Evolución del Bienestar General', fontweight='bold')
axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Análisis de patrones:")
correlacion_felicidad_estres = df_mood['felicidad'].corr(df_mood['estres'])
correlacion_motivacion_estres = df_mood['motivacion'].corr(df_mood['estres'])
print(f"Correlación Felicidad-Estrés: {correlacion_felicidad_estres:.3f}")
print(f"Correlación Motivación-Estrés: {correlacion_motivacion_estres:.3f}")
print(f"Día con mejor bienestar: Día {bienestar_diario.idxmax()}")
print(f"Día con menor bienestar: Día {bienestar_diario.idxmin()}")

In [ ]:
# Clustering de estados emocionales
print("CLUSTERING DE ESTADOS EMOCIONALES")
print("="*50)

# Preparar datos para clustering
features_clustering = ['felicidad', 'estres', 'motivacion', 'hora']
X_cluster = df_mood[features_clustering].copy()

# Escalar las características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Aplicar KMeans
n_clusters = 4  # Cuatro estados: mañana positiva, tarde productiva, noche relajada, momentos difíciles
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df_mood['cluster_estado'] = clusters

print(f"Se identificaron {n_clusters} clusters de estados emocionales")
print(f"Distribución de clusters:")
for i in range(n_clusters):
    count = sum(clusters == i)
    porcentaje = count / len(clusters) * 100
    print(f"  Cluster {i}: {count} registros ({porcentaje:.1f}%)")

# Analizar características de cada cluster
print(f"\nCARACTERÍSTICAS DE CADA CLUSTER:")
cluster_stats = df_mood.groupby('cluster_estado')[['felicidad', 'estres', 'motivacion', 'hora']].mean()

for i in range(n_clusters):
    stats = cluster_stats.loc[i]
    print(f"\nCluster {i}:")
    print(f"  Felicidad: {stats['felicidad']:.3f}")
    print(f"  Estrés: {stats['estres']:.3f}")
    print(f"  Motivación: {stats['motivacion']:.3f}")
    print(f"  Hora promedio: {stats['hora']:.1f}h")
    
    # Interpretación del cluster
    if stats['felicidad'] > 0.6 and stats['estres'] < 0.4:
        interpretacion = "Estado positivo y relajado"
    elif stats['motivacion'] > 0.6 and stats['estres'] < 0.5:
        interpretacion = "Estado productivo y enfocado"
    elif stats['estres'] > 0.6:
        interpretacion = "Estado de estrés elevado"
    else:
        interpretacion = "Estado emocional neutro"
    
    print(f"  Interpretación: {interpretacion}")

In [ ]:
# Modelo de predicción de estados emocionales
print("MODELO DE PREDICCIÓN DE ESTADOS EMOCIONALES")
print("="*60)

# Preparar datos para predicción
# Vamos a predecir el nivel de bienestar basado en hora, usuario y día
features_pred = ['usuario_id', 'dia', 'hora']
X_pred = df_mood[features_pred].values

# Variables objetivo: predecir cada estado emocional
# Convertir a clasificación categórica (alto/medio/bajo)
def categorizar_emocion(valor, tipo='normal'):
    if tipo == 'inverso':  # Para estrés (menos es mejor)
        if valor <= 0.4:
            return 0  # bajo (bueno)
        elif valor <= 0.6:
            return 1  # medio
        else:
            return 2  # alto (malo)
    else:  # Para felicidad y motivación (más es mejor)
        if valor <= 0.4:
            return 0  # bajo
        elif valor <= 0.6:
            return 1  # medio
        else:
            return 2  # alto

df_mood['felicidad_cat'] = df_mood['felicidad'].apply(categorizar_emocion)
df_mood['estres_cat'] = df_mood['estres'].apply(lambda x: categorizar_emocion(x, 'inverso'))
df_mood['motivacion_cat'] = df_mood['motivacion'].apply(categorizar_emocion)

# Entrenar modelos para cada emoción
emociones = {'felicidad': 'felicidad_cat', 'estres': 'estres_cat', 'motivacion': 'motivacion_cat'}
modelos = {}
accuracies = {}

for emocion, target in emociones.items():
    y = df_mood[target].values
    
    # Modelo Random Forest
    rf = RandomForestClassifier(n_estimators=50, random_state=42)
    
    # Validación cruzada
    if len(X_pred) >= 5:
        cv_scores = cross_val_score(rf, X_pred, y, cv=min(3, len(set(y))), scoring='accuracy')
        accuracy = cv_scores.mean()
        print(f"{emocion.capitalize()}: Accuracy = {accuracy:.3f} (+/- {cv_scores.std() * 2:.3f})")
    else:
        rf.fit(X_pred, y)
        accuracy = rf.score(X_pred, y)
        print(f"{emocion.capitalize()}: Accuracy = {accuracy:.3f} (dataset completo)")
    
    # Entrenar modelo final
    rf.fit(X_pred, y)
    modelos[emocion] = rf
    accuracies[emocion] = accuracy
    
    # Importancia de características
    importancias = rf.feature_importances_
    print(f"  Importancia: Usuario={importancias[0]:.3f}, Día={importancias[1]:.3f}, Hora={importancias[2]:.3f}")

print(f"\nAccuracy promedio del sistema: {np.mean(list(accuracies.values())):.3f}")

In [ ]:
# GRÁFICA 3: Visualización de clusters y predicciones
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Clusters en espacio felicidad-estrés
scatter = axes[0,0].scatter(df_mood['felicidad'], df_mood['estres'], 
                           c=df_mood['cluster_estado'], cmap='viridis', 
                           alpha=0.7, s=60)
axes[0,0].set_xlabel('Felicidad')
axes[0,0].set_ylabel('Estrés')
axes[0,0].set_title('Clusters de Estados Emocionales', fontweight='bold')
plt.colorbar(scatter, ax=axes[0,0], label='Cluster')

# 2. Estados emocionales por hora
estados_hora = df_mood.groupby(['hora', 'cluster_estado']).size().unstack(fill_value=0)
estados_hora.plot(kind='bar', stacked=True, ax=axes[0,1], colormap='Set3')
axes[0,1].set_xlabel('Hora del Día')
axes[0,1].set_ylabel('Número de Registros')
axes[0,1].set_title('Distribución de Estados por Hora', fontweight='bold')
axes[0,1].legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')

# 3. Evolución del bienestar por usuario
for usuario in df_mood['nombre'].unique():
    data = df_mood[df_mood['nombre'] == usuario]
    bienestar_usuario = data.groupby('dia')['bienestar'].mean()
    axes[1,0].plot(bienestar_usuario.index, bienestar_usuario.values, 
                   marker='o', label=usuario, linewidth=3)
axes[1,0].set_xlabel('Día')
axes[1,0].set_ylabel('Índice de Bienestar')
axes[1,0].set_title('Evolución del Bienestar por Usuario', fontweight='bold')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# 4. Matriz de confusión simplificada para felicidad
from sklearn.metrics import confusion_matrix
y_pred_felicidad = modelos['felicidad'].predict(X_pred)
cm = confusion_matrix(df_mood['felicidad_cat'], y_pred_felicidad)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1,1],
           xticklabels=['Bajo', 'Medio', 'Alto'],
           yticklabels=['Bajo', 'Medio', 'Alto'])
axes[1,1].set_xlabel('Predicción')
axes[1,1].set_ylabel('Real')
axes[1,1].set_title('Matriz de Confusión - Felicidad', fontweight='bold')

plt.tight_layout()
plt.show()

print("Insights de clustering:")
print(f"Total de clusters identificados: {n_clusters}")
print(f"Cluster más frecuente: {df_mood['cluster_estado'].mode().iloc[0]}")
print(f"Variabilidad emocional: {df_mood['bienestar'].std():.3f}")

In [ ]:
# RESUMEN FINAL DEL ANÁLISIS
print("HALLAZGOS CLAVE DEL ANÁLISIS DE MOODMAPS")
print("="*60)

# Estadísticas generales
print("ESTADÍSTICAS GENERALES:")
print(f"Total de registros emocionales: {len(df_mood)}")
print(f"Usuarios participantes: {df_mood['usuario_id'].nunique()}")
print(f"Período de seguimiento: {df_mood['dia'].nunique()} días")
print(f"Registros por día promedio: {len(df_mood) / df_mood['dia'].nunique():.1f}")

# Análisis por usuario
print(f"\nPATRONES POR USUARIO:")
for usuario_id in sorted(df_mood['usuario_id'].unique()):
    usuario_data = df_mood[df_mood['usuario_id'] == usuario_id]
    nombre = usuario_data['nombre'].iloc[0]
    
    felicidad_promedio = usuario_data['felicidad'].mean()
    estres_promedio = usuario_data['estres'].mean()
    motivacion_promedio = usuario_data['motivacion'].mean()
    bienestar_promedio = usuario_data['bienestar'].mean()
    
    print(f"\n  {nombre} ({len(usuario_data)} registros):")
    print(f"    Felicidad promedio: {felicidad_promedio:.3f}")
    print(f"    Estrés promedio: {estres_promedio:.3f}")
    print(f"    Motivación promedio: {motivacion_promedio:.3f}")
    print(f"    Índice de bienestar: {bienestar_promedio:.3f}")
    
    # Cluster más frecuente
    cluster_freq = usuario_data['cluster_estado'].mode().iloc[0]
    print(f"    Estado emocional predominante: Cluster {cluster_freq}")

# Patrones temporales
print(f"\nPATRONES TEMPORALES:")
mejor_hora = df_mood.groupby('hora')['bienestar'].mean().idxmax()
peor_hora = df_mood.groupby('hora')['bienestar'].mean().idxmin()
print(f"  Mejor hora emocional: {mejor_hora}:00h")
print(f"  Hora más desafiante: {peor_hora}:00h")

mejor_dia = df_mood.groupby('dia')['bienestar'].mean().idxmax()
peor_dia = df_mood.groupby('dia')['bienestar'].mean().idxmin()
print(f"  Mejor día del período: Día {mejor_dia}")
print(f"  Día más desafiante: Día {peor_dia}")

# Accuracies de los modelos
print(f"\nPRECISIÓN DE LOS MODELOS PREDICTIVOS:")
for emocion, accuracy in accuracies.items():
    print(f"  Predicción de {emocion}: {accuracy:.1%}")

accuracy_promedio = np.mean(list(accuracies.values()))
print(f"\nAccuracy promedio del sistema MoodMap: {accuracy_promedio:.1%}")

print(f"\nEl sistema puede predecir estados emocionales con {accuracy_promedio:.1%} de precisión")
print("basándose en patrones de usuario, día y hora")

In [ ]:
# GUARDAR MODELOS DE MOODMAPS
print("="*50)
print("GUARDANDO MODELOS DE MOODMAPS")
print("="*50)

# Crear directorio para modelos
models_dir = "modelos_entrenados"
os.makedirs(models_dir, exist_ok=True)

try:
    # 1. GUARDAR MODELO DE CLUSTERING DE ESTADOS EMOCIONALES
    clustering_file = os.path.join(models_dir, "modelo_clustering_moodmaps.pkl")
    scaler_file = os.path.join(models_dir, "scaler_moodmaps.pkl")
    
    joblib.dump(kmeans, clustering_file)
    joblib.dump(scaler, scaler_file)
    
    # 2. GUARDAR MODELOS DE PREDICCIÓN DE EMOCIONES
    modelos_pred_file = os.path.join(models_dir, "modelos_prediccion_moodmaps.pkl")
    joblib.dump(modelos, modelos_pred_file)
    
    # 3. GUARDAR METADATOS COMPLETOS
    metadata_moodmaps = {
        'feature_names_clustering': features_clustering,
        'feature_names_prediccion': features_pred,
        'n_clusters': n_clusters,
        'emociones_predichas': list(emociones.keys()),
        'accuracies': accuracies,
        'accuracy_promedio': accuracy_promedio,
        'n_samples': len(df_mood),
        'n_usuarios': df_mood['usuario_id'].nunique(),
        'n_dias': df_mood['dia'].nunique(),
        'model_types': {
            'clustering': 'KMeans',
            'prediccion': 'RandomForestClassifier'
        },
        'cluster_interpretations': {
            0: 'Estado emocional variante',
            1: 'Estado emocional estable',
            2: 'Estado emocional elevado',
            3: 'Estado emocional desafiante'
        },
        'categorias_emocionales': {
            0: 'Bajo',
            1: 'Medio', 
            2: 'Alto'
        },
        'creation_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'description': 'Modelos para clustering y predicción de estados emocionales en MoodMaps'
    }
    
    metadata_file = os.path.join(models_dir, "metadata_moodmaps.pkl")
    with open(metadata_file, 'wb') as f:
        pickle.dump(metadata_moodmaps, f)
    
    # 4. GUARDAR DATOS PROCESADOS
    datos_procesados_file = os.path.join(models_dir, "datos_moodmaps_procesados.pkl")
    df_mood.to_pickle(datos_procesados_file)
    
    print(f"Modelo de clustering guardado en: {clustering_file}")
    print(f"Scaler guardado en: {scaler_file}")
    print(f"Modelos de predicción guardados en: {modelos_pred_file}")
    print(f"Metadatos guardados en: {metadata_file}")
    print(f"Datos procesados guardados en: {datos_procesados_file}")
    print(f"Accuracy promedio: {metadata_moodmaps['accuracy_promedio']:.1%}")
    print(f"Muestras procesadas: {metadata_moodmaps['n_samples']}")
    print(f"Clusters identificados: {metadata_moodmaps['n_clusters']}")
    print(f"Emociones modeladas: {len(metadata_moodmaps['emociones_predichas'])}")
    
    # 5. EJEMPLO DE USO RÁPIDO
    print(f"\nEjemplo de predicción:")
    # Predecir estado emocional de usuario 1, día 5, hora 14
    ejemplo_pred = np.array([[1, 5, 14]])
    pred_felicidad = modelos['felicidad'].predict(ejemplo_pred)[0]
    pred_estres = modelos['estres'].predict(ejemplo_pred)[0]
    pred_motivacion = modelos['motivacion'].predict(ejemplo_pred)[0]
    
    categorias = ['Bajo', 'Medio', 'Alto']
    print(f"Usuario 1, día 5, 14:00h:")
    print(f"  Felicidad: {categorias[pred_felicidad]}")
    print(f"  Estrés: {categorias[pred_estres]}")
    print(f"  Motivación: {categorias[pred_motivacion]}")
    
    # Ejemplo de clustering
    ejemplo_cluster_input = np.array([[0.5, 0.3, 0.7, 14]])  # felicidad, estrés, motivación, hora
    ejemplo_cluster_scaled = scaler.transform(ejemplo_cluster_input)
    cluster_predicho = kmeans.predict(ejemplo_cluster_scaled)[0]
    interpretacion_cluster = metadata_moodmaps['cluster_interpretations'][cluster_predicho]
    print(f"\nEjemplo de clustering:")
    print(f"Estado (0.5, 0.3, 0.7, 14h) → Cluster {cluster_predicho}: {interpretacion_cluster}")
    
except Exception as e:
    print(f"Error al guardar modelos: {e}")
    print("Verificando variables necesarias...")
    print(f"df_mood existe: {'df_mood' in locals()}")
    print(f"modelos existe: {'modelos' in locals()}")
    print(f"kmeans existe: {'kmeans' in locals()}")